In [ ]:
### IMPORTS
import pandas as pd
import numpy as np
from backtester.pnl import pnl
from data_utils.io import load_partitioned_parquet
from data_utils.paths import make_data_path
from backtester.metrics.performance_report import PerformanceReport
from backtester.metrics.display import display_report

In [ ]:
###CONFIG PLEASE
INITAL_CAPITAL = 100000
DELAY_BARS = 1 # 1-bar execution delay
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2025'
END = '01/01/2026'


In [3]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


In [4]:
# pos = np.random.uniform(-1, 1, size=(len(mark_close)))
pos = np.random.normal(0, 0.5, size=len(mark_close))
pos = np.clip(pos, -1, 1)

In [ ]:
pnl_df = pnl(df_merge, pos, capital=INITAL_CAPITAL, delay_bars=DELAY_BARS)
print(pnl_df.tail())

                           asset_change (%)  signal (% of equity)  \
timestamp                                                           
2025-12-31 19:00:00+00:00         -0.001611             -0.743841   
2025-12-31 20:00:00+00:00          0.001425              0.179754   
2025-12-31 21:00:00+00:00          0.001501             -0.624829   
2025-12-31 22:00:00+00:00         -0.000740             -0.571697   
2025-12-31 23:00:00+00:00         -0.001081              0.727712   

                           held_pos (% of equity)  trade (% of equity)  \
timestamp                                                                
2025-12-31 19:00:00+00:00                0.382731            -0.492453   
2025-12-31 20:00:00+00:00               -0.743841            -1.126572   
2025-12-31 21:00:00+00:00                0.179754             0.923594   
2025-12-31 22:00:00+00:00               -0.624829            -0.804583   
2025-12-31 23:00:00+00:00               -0.571697             0.053132  

In [6]:
report=PerformanceReport(pnl_df)
display_report(report)


,Value
Core,
Start,2025-01-01 00:00
End,2025-12-31 23:00
Duration,364 days
Interval,1H
Bars,8760 bars
Starting Capital,"$100,000.00"
Final Capital,"$7,675.06"


,Value
Returns,
CAGR,"-92.32% ($-92,324.94)"
Net Return,"-92.32% ($-92,324.94)"
Gross Return,"5.41% ($5,410.79)"
Sharpe (per bar),-0.128
Sharpe (annualised),-11.998
Sortino,-16.354
Volatility,21.21%
Hit Rate (Trade),34.83%
Avg Win/Loss,1.162


,Value
Risk,
Max Drawdown,"-92.33% ($-92,329.61)"
Max DD Duration,8759.00 bars
Avg DD Duration,8759.00 bars
Time in Drawdown,99.99%
Calmar,-1.000
VaR 95%,-0.33% ($-334.26)
VaR 99%,-0.71% ($-705.80)
CVaR 95%,-0.57% ($-569.05)
CVaR 99%,"-1.01% ($-1,011.72)"


,Value
Costs,
Total Fee Drag,"262.01% ($262,013.22)"
Total Funding Drag,0.08% ($75.56)
Total Costs,"260.83% ($260,830.73)"
Fee % of Net,102.98%
Funding % of Net,0.03%
Cost % of Net,102.51%
Cost to Gross Ratio,34.776
Fee Drag (Sharpe),0.132
Funding Drag (Sharpe),-0.000


,Value
Position,
Avg Position Size,38.59%
Max Position Size,100.00%
Avg Long Size,38.82%
Avg Short Size,-38.36%
Time Long,50.33%
Time Short,49.66%
Time Flat,0.01%
Time in Market,99.99%
Avg Holding Period,1.00 bars
